# Lesson 6.1: The Logic of Hypothesis Testing

Find out how surprising 60 heads in 100 flips would be if the coin were fair, first by simulation and then with the exact binomial test.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## 1. Set up the coin result

Our data: 60 heads in 100 flips. The null hypothesis says the coin is fair (chance of heads = 0.5). NumPy makes the pretend coins and SciPy gives the exact test.

In [1]:
import numpy as np
from scipy import stats

flips = 100
heads = 60
rng = np.random.default_rng(42)   # seeded, so the result repeats

print('observed:', heads, 'heads in', flips, 'flips')

observed: 60 heads in 100 flips


## 2. Simulate a fair coin

We pretend the null is true and run 100,000 experiments of 100 fair flips. The p-value is the share of experiments at least as lopsided as ours, meaning 10 or more heads away from 50 in either direction.

In [2]:
sim = rng.binomial(n=flips, p=0.5, size=100_000)   # heads in each pretend experiment
gap = abs(heads - 50)                              # our result is 10 away from 50
extreme = np.abs(sim - 50) >= gap                  # too many OR too few heads
p_sim = extreme.mean()

print(f'average heads in the simulation: {sim.mean():.2f}')
print(f'simulated p-value = {p_sim:.4f}')

average heads in the simulation: 50.02
simulated p-value = 0.0571


## 3. Get the exact p-value

binomtest adds up the exact binomial chances, so no simulation is needed. The two-sided option counts both too many and too few heads. The simulated number should land close to the exact one.

In [3]:
result = stats.binomtest(heads, n=flips, p=0.5, alternative='two-sided')
ci = result.proportion_ci(confidence_level=0.95)

print(f'exact p-value = {result.pvalue:.4f}')
print(f'95% CI for the heads rate: ({ci.low:.3f}, {ci.high:.3f})')

exact p-value = 0.0569
95% CI for the heads rate: (0.497, 0.697)


## 4. Make the decision

Compare p with alpha = 0.05 and word the result carefully. The normal-curve shortcut (z = 2) gives a smaller p, which would wrongly say 'significant'. This shows why the method matters when a result sits near the cut-off.

In [4]:
alpha = 0.05
z = (heads - 50) / np.sqrt(flips * 0.5 * 0.5)
p_normal = 2 * stats.norm.sf(z)

print(f'normal shortcut: z = {z:.1f}, p = {p_normal:.4f}')
print('exact p <= alpha?', result.pvalue <= alpha)
if result.pvalue <= alpha:
    print('Reject H0: evidence the coin is not fair.')
else:
    print('Fail to reject H0: not enough evidence the coin is unfair.')

normal shortcut: z = 2.0, p = 0.0455
exact p <= alpha? False
Fail to reject H0: not enough evidence the coin is unfair.


## Try this

- Change heads to 62 and predict whether the p-value passes below 0.05, then run it.
- Change flips to 1000 and heads to 600 (still 60%). Predict what happens to the p-value, then run it.

## The same idea in R, SPSS and Stata

**R**

```r
binom.test(60, 100, p = 0.5)

# the simulation (random numbers, so close to but not exactly the Python value)
set.seed(42)
sim <- rbinom(100000, size = 100, prob = 0.5)
mean(abs(sim - 50) >= 10)
```

**SPSS** (Analyze > Nonparametric Tests > Legacy Dialogs > Binomial)

```
DATA LIST FREE / heads count.
BEGIN DATA
1 60
0 40
END DATA.
WEIGHT BY count.
NPAR TESTS
  /BINOMIAL (0.5)=heads.
```

**Stata**

```
bitesti 100 60 0.5
```